In [1]:
"""
International Trade Company Dataset Generator
=============================================
Company: GlobalTrade LLC, Moscow, Russia
Generates a realistic 3NF database suitable for:
  - Data cleaning tasks
  - BI dashboards
  - Anomaly detection
  - Fraud classification
  - Delivery time regression
  - Time series (port congestion, FX rates)
  - LSTM (purchase order state transitions)
"""

'\nInternational Trade Company Dataset Generator\n=============================================\nCompany: GlobalTrade LLC, Moscow, Russia\nGenerates a realistic 3NF database suitable for:\n  - Data cleaning tasks\n  - BI dashboards\n  - Anomaly detection\n  - Fraud classification\n  - Delivery time regression\n  - Time series (port congestion, FX rates)\n  - LSTM (purchase order state transitions)\n'

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta, date
import random
import os
import json
import warnings
warnings.filterwarnings("ignore")

In [3]:
fake_ru = Faker("ru_RU")
fake_en = Faker("en_US")
rng = np.random.default_rng(42)
random.seed(42)

In [4]:
OUTPUT_DIR = r"C:\Courses\Innopolis\2026\ВЭД\Данные2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = date(2019, 1, 1)
END_DATE   = date(2025, 12, 31)
N_DAYS     = (END_DATE - START_DATE).days

In [7]:
# ─────────────────────────────────────────────────────────────
# 1. COUNTRIES  (dim_countries)
# ─────────────────────────────────────────────────────────────
COUNTRIES = {
    # country_id: (name, region, country_risk_base, sanctions_risk, port_efficiency,
    #              avg_customs_days, currency_code, language, wto_member)
    "CN": ("China",          "Asia",          0.25, 0.20, 0.82, 3,  "CNY", "ZH", True),
    "DE": ("Germany",        "Europe",        0.05, 0.00, 0.90, 1,  "EUR", "DE", True),
    "TR": ("Turkey",         "Middle East",   0.30, 0.10, 0.65, 4,  "TRY", "TR", True),
    "IN": ("India",          "Asia",          0.28, 0.00, 0.60, 5,  "INR", "EN", True),
    "KR": ("South Korea",    "Asia",          0.10, 0.00, 0.88, 2,  "KRW", "KO", True),
    "IT": ("Italy",          "Europe",        0.12, 0.00, 0.75, 2,  "EUR", "IT", True),
    "PL": ("Poland",         "Europe",        0.10, 0.00, 0.80, 2,  "PLN", "PL", True),
    "AE": ("UAE",            "Middle East",   0.15, 0.05, 0.92, 1,  "AED", "AR", True),
    "VN": ("Vietnam",        "Asia",          0.22, 0.00, 0.58, 5,  "VND", "VI", True),
    "BR": ("Brazil",         "LatAm",         0.35, 0.00, 0.55, 6,  "BRL", "PT", True),
    "JP": ("Japan",          "Asia",          0.05, 0.00, 0.93, 1,  "JPY", "JA", True),
    "US": ("USA",            "North America", 0.05, 0.80, 0.88, 2,  "USD", "EN", True),
    "BY": ("Belarus",        "Europe",        0.55, 0.90, 0.40, 8,  "BYN", "RU", True),
    "KZ": ("Kazakhstan",     "Central Asia",  0.30, 0.05, 0.45, 7,  "KZT", "RU", True),
    "RU": ("Russia",         "Europe",        0.60, 0.95, 0.55, 6,  "RUB", "RU", True),
}

df_countries = pd.DataFrame.from_dict(
    COUNTRIES, orient="index",
    columns=["country_name","region","country_risk_base","sanctions_risk",
             "port_efficiency_base","avg_customs_days","currency_code","language","wto_member"]
).reset_index().rename(columns={"index":"country_id"})

In [8]:
# ─────────────────────────────────────────────────────────────
# 2. PRODUCT CATEGORIES & PRODUCTS  (dim_product_categories, dim_products)
# ─────────────────────────────────────────────────────────────
CATEGORIES = {
    1: ("Electronics",       "Electronic components, devices, consumer electronics"),
    2: ("Industrial Equip.", "Machinery, tools, industrial hardware"),
    3: ("Chemicals",         "Industrial chemicals, polymers, solvents"),
    4: ("Textiles",          "Fabrics, garments, raw fibers"),
    5: ("Automotive Parts",  "Spare parts, tires, accessories"),
    6: ("Food & Agro",       "Processed food, commodities, spices"),
    7: ("Pharma",            "Pharmaceuticals, medical supplies"),
    8: ("Packaging",         "Cardboard, plastics, labels"),
}

df_categories = pd.DataFrame.from_dict(
    CATEGORIES, orient="index",
    columns=["category_name","category_description"]
).reset_index().rename(columns={"index":"category_id"})

# Products: (name, category_id, unit, base_price_usd, weight_kg_per_unit, hs_code, perishable)
PRODUCTS_RAW = [
    ("Lithium Battery Cells",        1, "pcs",  12.50,  0.05, "850650", False),
    ("Industrial PCB Boards",        1, "pcs",  45.00,  0.30, "853400", False),
    ("LED Display Panels",           1, "pcs", 120.00,  2.50, "853110", False),
    ("Microcontrollers STM32",       1, "pcs",   3.20,  0.01, "854231", False),
    ("CCTV Camera Modules",          1, "pcs",  25.00,  0.40, "852580", False),
    ("CNC Machine Parts",            2, "pcs", 850.00, 15.00, "846599", False),
    ("Hydraulic Pumps",              2, "pcs", 320.00, 12.00, "841320", False),
    ("Industrial Bearings",          2, "pcs",  18.00,  0.80, "848210", False),
    ("Steel Welding Wire",           2, "kg",    2.80,  1.00, "831000", False),
    ("Pneumatic Cylinders",          2, "pcs",  95.00,  3.50, "841241", False),
    ("Epoxy Resin",                  3, "kg",    4.20,  1.00, "390691", False),
    ("Polyethylene Granules",        3, "kg",    1.65,  1.00, "390110", False),
    ("Industrial Solvents MEK",      3, "liter", 2.10,  0.82, "291412", False),
    ("PVC Compound",                 3, "kg",    2.30,  1.00, "390421", False),
    ("Cotton Yarn 32s",              4, "kg",    3.90,  1.00, "520512", False),
    ("Polyester Fabric 150gsm",      4, "meter", 1.80,  0.35, "540720", False),
    ("Workwear Garments",            4, "pcs",  14.50,  0.60, "621000", False),
    ("Non-woven Geotextile",         4, "sqm",   0.95,  0.12, "560390", False),
    ("Brake Pads Set",               5, "set",  22.00,  1.20, "870830", False),
    ("Automotive Filters Kit",       5, "set",  18.50,  0.80, "842131", False),
    ("Timing Belts",                 5, "pcs",   8.70,  0.25, "401120", False),
    ("Shock Absorbers",              5, "pcs",  45.00,  3.20, "870880", False),
    ("Sunflower Oil (bulk)",         6, "liter", 1.15,  0.92, "151211", True),
    ("Canned Fish (sardines)",       6, "pcs",   1.80,  0.35, "160413", True),
    ("Black Tea Assorted",           6, "kg",    8.50,  1.00, "090210", True),
    ("Dried Fruits Mix",             6, "kg",   12.00,  1.00, "081340", True),
    ("Paracetamol API",              7, "kg",   85.00,  1.00, "294220", False),
    ("Surgical Masks N95",           7, "pcs",   0.85,  0.01, "630790", False),
    ("Disposable Gloves (box)",      7, "box",  12.00,  0.50, "401511", False),
    ("IV Bags 500ml",                7, "pcs",   2.40,  0.55, "300590", True),
    ("Corrugated Cardboard Sheets",  8, "sqm",   0.65,  0.80, "481810", False),
    ("Stretch Film Rolls",           8, "roll",  8.20,  8.00, "392010", False),
]

df_products = pd.DataFrame(PRODUCTS_RAW,
    columns=["product_name","category_id","unit","base_price_usd",
             "weight_kg_per_unit","hs_code","perishable"])
df_products.insert(0, "product_id", range(1, len(df_products)+1))


In [11]:
# ─────────────────────────────────────────────────────────────
# 3. TRANSPORT MODES  (dim_transport_modes)
# ─────────────────────────────────────────────────────────────
TRANSPORT_MODES = [
    (1, "Sea FCL",    "Full Container Load by sea",       28, 1.0,  0.12,  True),
    (2, "Sea LCL",    "Less than Container Load by sea",  35, 1.0,  0.18,  True),
    (3, "Air Cargo",  "Air freight",                       5, 5.5,  0.05,  False),
    (4, "Rail",       "Rail freight (incl. Trans-Siberian)",14,1.8,  0.08,  False),
    (5, "Road",       "Truck / road freight",               7, 2.2,  0.10,  False),
    (6, "Multimodal", "Sea + Rail combination",            21, 1.3,  0.15,  True),
]

df_transport_modes = pd.DataFrame(TRANSPORT_MODES,
    columns=["transport_mode_id","mode_name","mode_description",
             "avg_transit_days","cost_multiplier","delay_probability","uses_port"])

In [12]:
# ─────────────────────────────────────────────────────────────
# 4. PORTS  (dim_ports)
# ─────────────────────────────────────────────────────────────
PORTS = [
    ("CNSHA", "Port of Shanghai",     "CN", 0.88, 45_000_000),
    ("CNTAO", "Port of Qingdao",      "CN", 0.82, 21_000_000),
    ("DEHAM", "Port of Hamburg",      "DE", 0.92,  9_700_000),
    ("TRIZM", "Port of Izmir",        "TR", 0.65,  1_500_000),
    ("INMUN", "JNPT Mumbai",          "IN", 0.60,  5_600_000),
    ("KRPUS", "Port of Busan",        "KR", 0.90, 22_000_000),
    ("ITGOA", "Port of Genoa",        "IT", 0.75,  2_800_000),
    ("AEJEA", "Port of Jebel Ali",    "AE", 0.92, 14_800_000),
    ("VNHPH", "Hai Phong Port",       "VN", 0.58,  5_000_000),
    ("BRSSZ", "Port of Santos",       "BR", 0.55, 4_000_000),
    ("JPYOK", "Port of Yokohama",     "JP", 0.93,  3_100_000),
    ("RUUNS", "Port of Ust-Luga",     "RU", 0.50,  1_900_000),
    ("RUNVS", "Port of Novorossiysk", "RU", 0.55,  2_800_000),
]

df_ports = pd.DataFrame(PORTS,
    columns=["port_id","port_name","country_id","efficiency_base","annual_teu_capacity"])


In [13]:
# ─────────────────────────────────────────────────────────────
# 5. SUPPLIERS  (dim_suppliers)
# ─────────────────────────────────────────────────────────────
SUPPLIER_POOL = [
    # (name, country_id, city, category_ids, is_fraudster)
    ("Shenzhen TechCore Ltd",      "CN", "Shenzhen",   [1],      False),
    ("Guangzhou ElecParts Co",     "CN", "Guangzhou",  [1,5],    False),
    ("Dongguan ManufactGroup",     "CN", "Dongguan",   [2,8],    False),
    ("Shanghai ChemTrade GmbH",    "CN", "Shanghai",   [3],      False),
    ("Yiwu TextileMaster",         "CN", "Yiwu",       [4],      False),
    ("Ningbo AutoSpares Corp",     "CN", "Ningbo",     [5],      False),
    ("Beijing PharmaSource",       "CN", "Beijing",    [7],      True),   # fraudster
    ("Siemens Industrial DE",      "DE", "Munich",     [2],      False),
    ("BASF Chemicals Europe",      "DE", "Ludwigshafen",[3],     False),
    ("Istanbul Textile Hub",       "TR", "Istanbul",   [4],      False),
    ("Ankara AutoParts TR",        "TR", "Ankara",     [5],      False),
    ("Mersin FoodExport TR",       "TR", "Mersin",     [6],      True),   # fraudster
    ("Tata Industrial IN",         "IN", "Mumbai",     [2,3],    False),
    ("Reliance ChemIN",            "IN", "Surat",      [3],      False),
    ("Samsung Elec Korea",         "KR", "Seoul",      [1],      False),
    ("Hyundai Parts KR",           "KR", "Ulsan",      [5],      False),
    ("Fiat Auto Parts IT",         "IT", "Turin",       [5],      False),
    ("Jebel Ali Trading",          "AE", "Dubai",      [6,7],    False),
    ("Vietnam GarmentPro",         "VN", "Ho Chi Minh",[4],      True),   # fraudster
    ("Brazil AgriTrade",           "BR", "Sao Paulo",  [6],      False),
    ("Toyota Logistics JP",        "JP", "Nagoya",     [5],      False),
    ("Canon Industrial JP",        "JP", "Tokyo",      [1,2],    False),
    ("Minsk MachineWorks BY",      "BY", "Minsk",      [2],      True),   # fraudster
    ("KazMunai Supply KZ",         "KZ", "Almaty",     [3],      False),
    ("PKN Orlen Chem PL",          "PL", "Warsaw",     [3,8],    False),
    ("Polish Textile Partners",    "PL", "Lodz",       [4],      False),
]

In [15]:
supplier_rows = []
for i, (name, cid, city, cat_ids, is_fraud) in enumerate(SUPPLIER_POOL, 1):
    reg_date = START_DATE - timedelta(days = int(rng.integers(180, 2500)))
    years_rel = round((START_DATE - reg_date).days / 365, 1)
    reliability = round(rng.beta(8 if not is_fraud else 3, 2), 3)
    maturity = rng.integers(2, 20)
    payment_terms = random.choice([30, 45, 60, 90])
    min_order_usd = int(rng.integers(500, 10000) / 500) * 500
    certifications = random.sample(["ISO9001","ISO14001","CE","FDA","HALAL","KOSHER","GMP"],
                                   k=rng.integers(1, 4))
    supplier_rows.append({
        "supplier_id": i,
        "supplier_name": name,
        "country_id": cid,
        "city": city,
        "primary_category_id": cat_ids[0],
        "registration_date": reg_date,
        "years_of_relationship": years_rel,
        "reliability_score": reliability,
        "maturity_score": maturity,
        "payment_terms_days": payment_terms,
        "min_order_usd": min_order_usd,
        "certifications": json.dumps(certifications),
        "is_fraudster": is_fraud,
        "active": True,
    })

df_suppliers = pd.DataFrame(supplier_rows)

# Supplier ↔ Product mapping  (dim_supplier_products)
sp_rows = []
for _, sup in df_suppliers.iterrows():
    cat = sup["primary_category_id"]
    prods = df_products[df_products["category_id"] == cat]["product_id"].tolist()
    for pid in prods:
        price_var = rng.uniform(0.85, 1.20)
        sp_rows.append({
            "supplier_id": int(sup["supplier_id"]),
            "product_id": int(pid),
            "supplier_price_usd": round(float(df_products.loc[df_products["product_id"]==pid,"base_price_usd"].values[0]) * price_var, 2),
            "lead_time_days": int(rng.integers(7, 45)),
            "moq": int(rng.integers(50, 500)),
        })

df_supplier_products = pd.DataFrame(sp_rows).drop_duplicates(["supplier_id","product_id"])
df_supplier_products.insert(0, "sp_id", range(1, len(df_supplier_products)+1))

In [16]:
# ─────────────────────────────────────────────────────────────
# 6. CUSTOMERS  (dim_customers) — all in Russia
# ─────────────────────────────────────────────────────────────
RU_CITIES = [
    ("Moscow", 55.75, 37.62), ("Saint Petersburg", 59.93, 30.32),
    ("Novosibirsk", 54.98, 82.90), ("Yekaterinburg", 56.83, 60.60),
    ("Kazan", 55.79, 49.12), ("Nizhny Novgorod", 56.33, 44.00),
    ("Krasnoyarsk", 56.02, 92.87), ("Chelyabinsk", 55.15, 61.40),
    ("Omsk", 54.98, 73.37), ("Samara", 53.20, 50.15),
    ("Rostov-on-Don", 47.23, 39.72), ("Ufa", 54.74, 55.97),
    ("Volgograd", 48.72, 44.50), ("Perm", 58.01, 56.25),
    ("Voronezh", 51.67, 39.18),
]

INDUSTRIES = ["Manufacturing","Retail","Construction","Healthcare",
              "Food Processing","Automotive","Electronics Assembly","Wholesale"]

customer_rows = []
for i in range(1, 61):
    city_data = random.choice(RU_CITIES)
    industry  = random.choice(INDUSTRIES)
    segment   = random.choice(["Enterprise","SME","Micro"])
    annual_rev = {"Enterprise": rng.integers(500, 5000),
                  "SME":        rng.integers(50,  500),
                  "Micro":      rng.integers(5,   50)}[segment]
    customer_rows.append({
        "customer_id": i,
        "customer_name": fake_ru.company(),
        "city": city_data[0],
        "latitude": city_data[1] + rng.uniform(-0.3, 0.3),
        "longitude": city_data[2] + rng.uniform(-0.3, 0.3),
        "industry": industry,
        "segment": segment,
        "annual_revenue_mln_rub": int(annual_rev),
        "credit_limit_usd": int(rng.integers(10, 500) * 1000),
        "registration_date": START_DATE - timedelta(days=int(rng.integers(180, 3650))),
        "inn": fake_ru.numerify("##########"),   # Russian tax ID
    })

df_customers = pd.DataFrame(customer_rows)

In [17]:
# ─────────────────────────────────────────────────────────────
# 7. DAILY MACRO TIME SERIES
# ─────────────────────────────────────────────────────────────
dates = [START_DATE + timedelta(days=d) for d in range(N_DAYS + 1)]
n = len(dates)

# --- Macro shocks (affect FX, risk, port congestion) ---
shocks = np.zeros(n)
SHOCK_EVENTS = [
    # (start_day_offset, duration_days, intensity, description)
    (365,   90,  0.8, "COVID_wave_1"),        # 2020 Q1-Q2
    (550,   60,  0.5, "COVID_wave_2"),        # 2020 Q4
    (730,   45,  0.6, "Suez_blockage_ripple"),# 2021 Q1
    (920,   30,  0.4, "Supply_chain_crunch"), # 2021 mid
    (1140, 200,  1.0, "Russia_sanctions"),    # 2022 Feb+ (most severe)
    (1340,  60,  0.6, "Energy_crisis"),       # 2022 Aug
    (1550,  45,  0.3, "Banking_mini_crisis"), # 2023 Mar
    (1700,  30,  0.25,"Middle_East_tension"), # 2023-24
]
for start, dur, intensity, _ in SHOCK_EVENTS:
    if start < n:
        end = min(start + dur, n)
        ramp = np.concatenate([
            np.linspace(0, intensity, dur//3),
            np.ones(max(0, dur - 2*(dur//3))) * intensity,
            np.linspace(intensity, 0, dur//3)
        ])[:end-start]
        shocks[start:end] += ramp

shocks = np.clip(shocks, 0, 1.5)

# --- FX Rates (daily, USD base) ---
def simulate_fx(base_rate, vol, trend_per_year=0.0, shock_sensitivity=1.0):
    """GBM-like with shock impact"""
    dt = 1/252
    log_returns = rng.normal(trend_per_year/252, vol*np.sqrt(dt), n)
    log_returns += shock_sensitivity * shocks * rng.normal(0, 0.015, n)
    prices = base_rate * np.exp(np.cumsum(log_returns))
    return np.round(prices, 4)

fx_data = {
    "date": dates,
    "USDRUB": simulate_fx(73.0,  0.18, 0.08,  2.0),   # RUB depreciates, shock-sensitive
    "USDCNY": simulate_fx(6.45,  0.04, 0.01,  0.3),
    "USDEUR": simulate_fx(0.85,  0.06,-0.005, 0.2),
    "USDTRY": simulate_fx(8.5,   0.22, 0.25,  0.8),   # high inflation Turkey
    "USDINR": simulate_fx(73.5,  0.05, 0.03,  0.2),
    "USDKRW": simulate_fx(1180,  0.07, 0.01,  0.3),
    "USDVND": simulate_fx(22800, 0.03, 0.01,  0.1),
    "USDBRL": simulate_fx(5.4,   0.15, 0.06,  0.5),
    "USDJPY": simulate_fx(108,   0.07,-0.02,  0.3),
    "USDKZT": simulate_fx(430,   0.10, 0.04,  0.8),
    "USDBYN": simulate_fx(2.6,   0.12, 0.05,  1.2),
    "USDAED": simulate_fx(3.672, 0.003, 0.0,  0.0),  # pegged
    "USDPLN": simulate_fx(3.9,   0.08, 0.01,  0.3),
}

df_fx = pd.DataFrame(fx_data)
df_fx.insert(0, "fx_id", range(1, len(df_fx)+1))

# --- Port Congestion Index (daily per port, 0-1) ---
port_congestion_rows = []
pid_list = df_ports["port_id"].tolist()

for port_id in pid_list:
    port_row = df_ports[df_ports["port_id"]==port_id].iloc[0]
    eff = port_row["efficiency_base"]

    # Seasonal pattern: higher congestion in Q4 (peak season) and Chinese New Year
    day_of_year = np.array([(d.timetuple().tm_yday) for d in dates])
    seasonal = (0.15 * np.sin(2 * np.pi * (day_of_year - 30) / 365)    # Chinese New Year dip ~day30
              + 0.20 * np.sin(2 * np.pi * (day_of_year - 290) / 365))  # Q4 peak ~day290
    base_congestion = (1 - eff) + seasonal * 0.5
    base_congestion += 0.3 * shocks  # macro shocks inflate congestion
    noise = rng.normal(0, 0.05, n)
    congestion = np.clip(base_congestion + noise, 0.0, 1.0)

    # Suez special spike for sea ports in 2021 Q1
    if port_row["country_id"] not in ["RU","KZ","BY"]:
        suez_day = 455  # approx March 2021
        congestion[suez_day:suez_day+30] = np.clip(
            congestion[suez_day:suez_day+30] + rng.uniform(0.3, 0.5), 0, 1)

    for d_idx, d in enumerate(dates):
        port_congestion_rows.append({
            "date": d,
            "port_id": port_id,
            "congestion_index": round(float(congestion[d_idx]), 4),
            "vessels_waiting": max(0, int(congestion[d_idx] * 80 + rng.normal(0, 5))),
        })

df_port_congestion = pd.DataFrame(port_congestion_rows)
df_port_congestion.insert(0, "pc_id", range(1, len(df_port_congestion)+1))

# --- Country Risk (quarterly) ---
country_risk_rows = []
quarters = pd.date_range(START_DATE.isoformat(), END_DATE.isoformat(), freq="QS")
for q in quarters:
    q_date = q.date()
    d_idx = min((q_date - START_DATE).days, n-1)
    for _, cr in df_countries.iterrows():
        base = cr["country_risk_base"]
        shock_bump = 0.3 * shocks[d_idx]
        # sanctions especially hit RU/BY in 2022+
        if cr["country_id"] in ["RU","BY"] and q_date >= date(2022, 3, 1):
            shock_bump += 0.3
        risk = round(np.clip(base + shock_bump + rng.normal(0, 0.02), 0, 1), 3)
        country_risk_rows.append({
            "country_id": cr["country_id"],
            "quarter_start": q_date,
            "country_risk_score": risk,
            "sanctions_risk_score": round(float(cr["sanctions_risk"]) +
                (0.4 if cr["country_id"] in ["RU","BY","US"] and q_date >= date(2022,3,1) else 0) +
                rng.normal(0, 0.01), 3),
            "political_stability": round(np.clip(1 - risk + rng.normal(0, 0.03), 0, 1), 3),
        })

df_country_risk = pd.DataFrame(country_risk_rows)
df_country_risk["sanctions_risk_score"] = df_country_risk["sanctions_risk_score"].clip(0, 1)
df_country_risk.insert(0, "cr_id", range(1, len(df_country_risk)+1))

In [20]:
# ─────────────────────────────────────────────────────────────
# 8. PURCHASE ORDERS  (fact_purchase_orders)
#    Covers: fraud detection, regression (delivery time), LSTM (state transitions)
# ─────────────────────────────────────────────────────────────
PO_STATES = ["Draft","Confirmed","Supplier_Processing","Shipped",
             "In_Transit","Customs_Clearance","Delivered","Cancelled"]

def get_fx_rate(date_val, currency):
    col = f"USD{currency}"
    if col not in df_fx.columns:
        return 1.0
    d_idx = min((date_val - START_DATE).days, n-1)
    return float(df_fx.iloc[d_idx][col])

def get_port_congestion(date_val, port_id):
    if port_id is None:
        return 0.0
    row = df_port_congestion[
        (df_port_congestion["date"] == date_val) &
        (df_port_congestion["port_id"] == port_id)
    ]
    return float(row["congestion_index"].values[0]) if len(row) > 0 else 0.2

def _build_state_sequence(order_date, final_state, delivery_days, reliability, congestion):
    """Generate realistic state timestamps for LSTM training."""
    events = []
    d = order_date

    confirm_lag = int(rng.integers(1, 5))
    events.append(("Draft", d, confirm_lag))
    d += timedelta(days=confirm_lag)

    if final_state == "Cancelled" and delivery_days is None:
        # Phantom: cancelled at confirmed
        events.append(("Confirmed", d, 0))
        events.append(("Cancelled", d, 0))
        return events

    proc_days = int(rng.integers(3, max(4, int((1 - reliability)*15 + 3))))
    events.append(("Confirmed", d, proc_days))
    d += timedelta(days=proc_days)

    ship_days = int(rng.integers(1, 4))
    events.append(("Supplier_Processing", d, ship_days))
    d += timedelta(days=ship_days)

    if final_state == "Cancelled":
        events.append(("Shipped", d, 0))
        events.append(("Cancelled", d, 0))
        return events

    transit_days = max(1, (delivery_days or 14) - proc_days - ship_days - 5)
    events.append(("Shipped", d, transit_days))
    d += timedelta(days=transit_days)

    in_transit = int(rng.integers(1, 4))
    events.append(("In_Transit", d, in_transit))
    d += timedelta(days=in_transit)

    customs_days = max(1, int(rng.integers(1, 8) + congestion * 5))
    events.append(("Customs_Clearance", d, customs_days))
    d += timedelta(days=customs_days)

    events.append(("Delivered", d, 0))
    return events

# Build PO state transition events table separately
po_rows = []
po_state_rows = []

po_id = 1
state_event_id = 1

# Generate ~2800 purchase orders over 6 years
order_dates = sorted([START_DATE + timedelta(days=int(d))
                      for d in rng.integers(0, N_DAYS, size=2800)])

for order_date in order_dates:
    # Pick supplier
    sup = df_suppliers.sample(1).iloc[0]
    sup_id = int(sup["supplier_id"])
    sup_country = sup["country_id"]
    is_fraud = bool(sup["is_fraudster"])

    # Pick product from supplier's catalog
    sup_prods = df_supplier_products[df_supplier_products["supplier_id"]==sup_id]
    if len(sup_prods) == 0:
        continue
    sp_row = sup_prods.sample(1).iloc[0]
    prod_id = int(sp_row["product_id"])
    prod = df_products[df_products["product_id"]==prod_id].iloc[0]

    # Transport mode (influenced by country/product)
    if sup_country in ["DE","PL","BY","KZ"]:
        mode_weights = [0.1, 0.05, 0.10, 0.45, 0.25, 0.05]  # rail/road dominant
    elif sup_country in ["CN","VN","KR","JP"]:
        mode_weights = [0.45, 0.15, 0.15, 0.15, 0.05, 0.05]  # sea dominant
    else:
        mode_weights = [0.35, 0.15, 0.20, 0.10, 0.15, 0.05]

    mode_id = int(rng.choice(range(1,7), p=mode_weights/np.sum(mode_weights)))
    mode = df_transport_modes[df_transport_modes["transport_mode_id"]==mode_id].iloc[0]

    # Port assignment
    uses_port = bool(mode["uses_port"])
    origin_port_id = None
    dest_port_id   = None
    if uses_port:
        country_ports = df_ports[df_ports["country_id"]==sup_country]["port_id"].tolist()
        if country_ports:
            origin_port_id = random.choice(country_ports)
        ru_ports = df_ports[df_ports["country_id"]=="RU"]["port_id"].tolist()
        dest_port_id = random.choice(ru_ports)

    # Quantity and pricing
    qty = int(rng.integers(int(sp_row["moq"]), int(sp_row["moq"]) * 5))
    unit_price_usd = float(sp_row["supplier_price_usd"])
    if is_fraud:
        # Fraudsters may invoice much higher or much lower (bait & switch)
        fraud_type = random.choice(["overinvoice","underinvoice","phantom"])
        if fraud_type == "overinvoice":
            unit_price_usd *= rng.uniform(1.3, 2.5)
        elif fraud_type == "underinvoice":
            unit_price_usd *= rng.uniform(0.2, 0.6)
        # phantom: normal price but goods never arrive
    else:
        fraud_type = None

    total_usd = round(qty * unit_price_usd, 2)
    currency   = COUNTRIES[sup_country][6]
    fx_rate    = get_fx_rate(order_date, currency)
    total_local = round(total_usd * fx_rate, 2)

    # Delivery time calculation
    d_idx = min((order_date - START_DATE).days, n-1)
    base_transit = int(mode["avg_transit_days"])
    shock_delay  = int(shocks[d_idx] * 15)
    cust_delay   = int(COUNTRIES[sup_country][5])  # avg_customs_days

    port_cong = 0.0
    if origin_port_id:
        port_cong = get_port_congestion(order_date, origin_port_id)
        shock_delay += int(port_cong * 20)

    sup_reliability = float(sup["reliability_score"])
    reliability_delay = int((1 - sup_reliability) * 10)

    if is_fraud and fraud_type == "phantom":
        delivered = False
        actual_delivery_days = None
        final_state = "Cancelled"
    else:
        noise = int(rng.normal(0, 3))
        actual_delivery_days = max(3, base_transit + shock_delay + cust_delay
                                   + reliability_delay + noise)
        delivered = True
        final_state = "Delivered"
        # Random cancellations (~4%)
        if rng.random() < 0.04:
            delivered = False
            actual_delivery_days = None
            final_state = "Cancelled"

    # Incoterm
    incoterm = random.choice(["EXW","FCA","FOB","CIF","DAP","DDP"])
    payment_method = random.choice(["Wire Transfer","L/C","Documentary Collection","Open Account"])

    insurance_pct = rng.uniform(0.003, 0.012)
    freight_usd = round(total_usd * float(mode["cost_multiplier"]) * 0.08
                        * (1 + port_cong * 0.3), 2)
    insurance_usd = round(total_usd * insurance_pct, 2)
    customs_duty_pct = rng.uniform(0.03, 0.20)
    customs_duty_usd = round(total_usd * customs_duty_pct, 2)

    po_rows.append({
        "po_id":                po_id,
        "order_date":           order_date,
        "supplier_id":          sup_id,
        "product_id":           prod_id,
        "transport_mode_id":    mode_id,
        "origin_port_id":       origin_port_id,
        "destination_port_id":  dest_port_id,
        "quantity":             qty,
        "unit_price_usd":       round(unit_price_usd, 2),
        "total_value_usd":      total_usd,
        "currency_code":        currency,
        "fx_rate_to_usd":       fx_rate,
        "total_value_local":    total_local,
        "freight_cost_usd":     freight_usd,
        "insurance_cost_usd":   insurance_usd,
        "customs_duty_usd":     customs_duty_usd,
        "incoterm":             incoterm,
        "payment_method":       payment_method,
        "expected_delivery_days": base_transit + cust_delay,
        "actual_delivery_days": actual_delivery_days,
        "port_congestion_at_order": round(port_cong, 4),
        "shock_index_at_order": round(float(shocks[d_idx]), 4),
        "supplier_reliability": round(sup_reliability, 3),
        "final_state":          final_state,
        "is_fraud":             is_fraud,
        "fraud_type":           fraud_type,
    })

    # State transition log for LSTM
    states_seq = _build_state_sequence(order_date, final_state, actual_delivery_days,
                                        sup_reliability, port_cong)
    for s_name, s_date, s_duration in states_seq:
        po_state_rows.append({
            "state_event_id": state_event_id,
            "po_id":          po_id,
            "state":          s_name,
            "state_date":     s_date,
            "duration_days":  s_duration,
        })
        state_event_id += 1

    po_id += 1




# Rebuild with function defined
po_rows = []
po_state_rows = []
po_id = 1
state_event_id = 1

for order_date in order_dates:
    sup = df_suppliers.sample(1).iloc[0]
    sup_id = int(sup["supplier_id"])
    sup_country = sup["country_id"]
    is_fraud = bool(sup["is_fraudster"])

    sup_prods = df_supplier_products[df_supplier_products["supplier_id"]==sup_id]
    if len(sup_prods) == 0:
        continue
    sp_row = sup_prods.sample(1).iloc[0]
    prod_id = int(sp_row["product_id"])

    if sup_country in ["DE","PL","BY","KZ"]:
        mode_weights = np.array([0.10, 0.05, 0.10, 0.45, 0.25, 0.05])
    elif sup_country in ["CN","VN","KR","JP"]:
        mode_weights = np.array([0.45, 0.15, 0.15, 0.15, 0.05, 0.05])
    else:
        mode_weights = np.array([0.35, 0.15, 0.20, 0.10, 0.15, 0.05])

    mode_id = int(rng.choice(range(1,7), p=mode_weights/mode_weights.sum()))
    mode = df_transport_modes[df_transport_modes["transport_mode_id"]==mode_id].iloc[0]

    uses_port = bool(mode["uses_port"])
    origin_port_id = None
    dest_port_id   = None
    if uses_port:
        country_ports = df_ports[df_ports["country_id"]==sup_country]["port_id"].tolist()
        if country_ports:
            origin_port_id = random.choice(country_ports)
        ru_ports = df_ports[df_ports["country_id"]=="RU"]["port_id"].tolist()
        dest_port_id = random.choice(ru_ports)

    qty = int(rng.integers(int(sp_row["moq"]), int(sp_row["moq"]) * 5))
    unit_price_usd = float(sp_row["supplier_price_usd"])

    fraud_type = None
    if is_fraud:
        fraud_type = random.choice(["overinvoice","underinvoice","phantom"])
        if fraud_type == "overinvoice":
            unit_price_usd *= rng.uniform(1.3, 2.5)
        elif fraud_type == "underinvoice":
            unit_price_usd *= rng.uniform(0.2, 0.6)

    total_usd = round(qty * unit_price_usd, 2)
    currency   = COUNTRIES[sup_country][6]
    fx_rate    = get_fx_rate(order_date, currency)
    total_local = round(total_usd * fx_rate, 2)

    d_idx = min((order_date - START_DATE).days, n-1)
    base_transit = int(mode["avg_transit_days"])
    shock_delay  = int(shocks[d_idx] * 15)
    cust_delay   = int(COUNTRIES[sup_country][5])
    port_cong = 0.0
    if origin_port_id:
        port_cong = get_port_congestion(order_date, origin_port_id)
        shock_delay += int(port_cong * 20)

    sup_reliability = float(sup["reliability_score"])
    reliability_delay = int((1 - sup_reliability) * 10)

    if is_fraud and fraud_type == "phantom":
        delivered = False
        actual_delivery_days = None
        final_state = "Cancelled"
    else:
        noise = int(rng.normal(0, 3))
        actual_delivery_days = max(3, base_transit + shock_delay + cust_delay
                                   + reliability_delay + noise)
        delivered = True
        final_state = "Delivered"
        if rng.random() < 0.04:
            delivered = False
            actual_delivery_days = None
            final_state = "Cancelled"

    incoterm = random.choice(["EXW","FCA","FOB","CIF","DAP","DDP"])
    payment_method = random.choice(["Wire Transfer","L/C","Documentary Collection","Open Account"])
    insurance_pct  = rng.uniform(0.003, 0.012)
    freight_usd    = round(total_usd * float(mode["cost_multiplier"]) * 0.08
                           * (1 + port_cong * 0.3), 2)
    insurance_usd  = round(total_usd * insurance_pct, 2)
    customs_duty_pct = rng.uniform(0.03, 0.20)
    customs_duty_usd = round(total_usd * customs_duty_pct, 2)

    po_rows.append({
        "po_id":                po_id,
        "order_date":           order_date,
        "supplier_id":          sup_id,
        "product_id":           prod_id,
        "transport_mode_id":    mode_id,
        "origin_port_id":       origin_port_id,
        "destination_port_id":  dest_port_id,
        "quantity":             qty,
        "unit_price_usd":       round(unit_price_usd, 2),
        "total_value_usd":      total_usd,
        "currency_code":        currency,
        "fx_rate_to_usd":       fx_rate,
        "total_value_local":    total_local,
        "freight_cost_usd":     freight_usd,
        "insurance_cost_usd":   insurance_usd,
        "customs_duty_usd":     customs_duty_usd,
        "incoterm":             incoterm,
        "payment_method":       payment_method,
        "expected_delivery_days": base_transit + cust_delay,
        "actual_delivery_days": actual_delivery_days,
        "port_congestion_at_order": round(port_cong, 4),
        "shock_index_at_order": round(float(shocks[d_idx]), 4),
        "supplier_reliability": round(sup_reliability, 3),
        "final_state":          final_state,
        "is_fraud":             is_fraud,
        "fraud_type":           fraud_type,
    })

    states_seq = _build_state_sequence(order_date, final_state, actual_delivery_days,
                                        sup_reliability, port_cong)
    for s_name, s_date, s_duration in states_seq:
        po_state_rows.append({
            "state_event_id": state_event_id,
            "po_id":          po_id,
            "state":          s_name,
            "state_date":     s_date,
            "duration_days":  s_duration,
        })
        state_event_id += 1

    po_id += 1

df_po = pd.DataFrame(po_rows)
df_po_states = pd.DataFrame(po_state_rows)

print(f"Purchase orders generated: {len(df_po)}")
print(f"Fraud rate: {df_po['is_fraud'].mean():.2%}")
print(f"PO state events: {len(df_po_states)}")

Purchase orders generated: 2800
Fraud rate: 15.54%
PO state events: 18712


In [21]:
# ─────────────────────────────────────────────────────────────
# 9. SALES ORDERS  (fact_sales_orders)
# ─────────────────────────────────────────────────────────────
so_rows = []
so_id = 1

delivered_pos = df_po[df_po["final_state"] == "Delivered"].copy()

for _, po in delivered_pos.iterrows():
    # Each delivered PO may generate 1-3 sales orders
    n_sales = int(rng.integers(1, 4))
    rem_qty = int(po["quantity"])
    base_sell_usd = float(po["unit_price_usd"]) * rng.uniform(1.15, 1.45)  # markup
    sell_date = po["order_date"] + timedelta(
        days=int(po["actual_delivery_days"] or 14) + int(rng.integers(1, 30)))

    for s in range(n_sales):
        if rem_qty <= 0:
            break
        cust = df_customers.sample(1).iloc[0]
        frac = rng.uniform(0.2, 0.8) if s < n_sales - 1 else 1.0
        s_qty = max(1, int(rem_qty * frac))
        rem_qty -= s_qty

        sale_usd = round(s_qty * base_sell_usd, 2)
        rub_rate = get_fx_rate(sell_date, "RUB")
        sale_rub = round(sale_usd * rub_rate, 2)
        payment_days = random.choice([0, 14, 30, 45, 60])
        discount = round(rng.choice([0, 0, 0, 0.02, 0.05, 0.10],
                                     p=[0.50, 0.15, 0.12, 0.10, 0.08, 0.05]), 2)

        so_rows.append({
            "so_id":            so_id,
            "sale_date":        sell_date + timedelta(days=int(rng.integers(0, 20))),
            "po_id":            int(po["po_id"]),
            "customer_id":      int(cust["customer_id"]),
            "product_id":       int(po["product_id"]),
            "quantity":         s_qty,
            "unit_price_usd":   round(base_sell_usd, 2),
            "discount_pct":     discount,
            "total_value_usd":  round(sale_usd * (1 - discount), 2),
            "total_value_rub":  round(sale_rub * (1 - discount), 2),
            "fx_rate_usd_rub":  rub_rate,
            "payment_terms_days": payment_days,
            "delivery_city":    cust["city"],
        })
        so_id += 1

df_so = pd.DataFrame(so_rows)
print(f"Sales orders generated: {len(df_so)}")

Sales orders generated: 5171


In [22]:
# ─────────────────────────────────────────────────────────────
# 10. SUPPLIER AUDIT LOG  (fact_supplier_audits)
# ─────────────────────────────────────────────────────────────
audit_rows = []
audit_id = 1
for _, sup in df_suppliers.iterrows():
    n_audits = int(rng.integers(1, 5))
    for _ in range(n_audits):
        a_date = START_DATE + timedelta(days=int(rng.integers(0, N_DAYS)))
        findings = int(rng.integers(0, 3)) if not sup["is_fraudster"] else int(rng.integers(2, 8))
        score = round(rng.beta(8 if not sup["is_fraudster"] else 3, 2), 3)
        audit_rows.append({
            "audit_id":        audit_id,
            "supplier_id":     int(sup["supplier_id"]),
            "audit_date":      a_date,
            "audit_type":      random.choice(["Financial","Quality","Compliance","On-site"]),
            "auditor":         fake_en.name(),
            "findings_count":  findings,
            "audit_score":     score,
            "passed":          score >= 0.6,
            "notes":           "" if findings == 0 else random.choice([
                "Minor documentation gaps", "Price discrepancy noted",
                "Delivery delays observed", "Certificate expired",
                "Subcontractor used without disclosure", "Invoice mismatch",
            ]),
        })
        audit_id += 1

df_audits = pd.DataFrame(audit_rows)

In [23]:
# ─────────────────────────────────────────────────────────────
# 11. INJECT DATA QUALITY ISSUES (for cleaning tasks)
# ─────────────────────────────────────────────────────────────
print("Injecting data quality issues...")

def inject_issues(df, target_col, pct_null=0.03, pct_outlier=0.01, pct_dup=0.005):
    df = df.copy()
    n_rows = len(df)

    # Missing values
    null_idx = rng.choice(n_rows, size=int(n_rows * pct_null), replace=False)
    df.loc[null_idx, target_col] = np.nan

    # Outliers in numeric cols
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols and pct_outlier > 0:
        out_col = random.choice(num_cols)
        out_idx = rng.choice(n_rows, size=max(1, int(n_rows * pct_outlier)), replace=False)
        df.loc[out_idx, out_col] *= rng.uniform(5, 20, size=len(out_idx))

    # Duplicates
    dup_count = max(1, int(n_rows * pct_dup))
    dup_rows  = df.sample(dup_count, random_state=0)
    df = pd.concat([df, dup_rows], ignore_index=True)

    return df

df_po   = inject_issues(df_po,   "actual_delivery_days", 0.04, 0.015, 0.008)
df_so   = inject_issues(df_so,   "total_value_usd",       0.03, 0.010, 0.005)
df_suppliers = inject_issues(df_suppliers, "reliability_score", 0.02, 0.00, 0.003)

# Typos in categorical
typo_map = {"Sea FCL": "SEA FCL", "Air Cargo": "air cargo", "Road": "road "}
mode_col = df_po["transport_mode_id"].copy()
typo_idx = rng.choice(len(df_po), size=int(len(df_po)*0.01), replace=False)
# (We store mode_id as int, so inject string typos in a text column instead)
df_po.loc[typo_idx, "incoterm"] = df_po.loc[typo_idx, "incoterm"].str.lower()

# Impossible value: negative quantity in a few rows
neg_idx = rng.choice(len(df_po), size=5, replace=False)
df_po.loc[neg_idx, "quantity"] = -df_po.loc[neg_idx, "quantity"]

# Date format inconsistency: convert some dates to string with different format
str_idx = rng.choice(len(df_so), size=int(len(df_so)*0.01), replace=False)
df_so["sale_date"] = df_so["sale_date"].astype(str)
for i in str_idx:
    if i < len(df_so):
        d = df_so.loc[i, "sale_date"]
        try:
            dt = datetime.strptime(str(d)[:10], "%Y-%m-%d")
            df_so.loc[i, "sale_date"] = dt.strftime("%d/%m/%Y")
        except:
            pass

Injecting data quality issues...


In [24]:
# ─────────────────────────────────────────────────────────────
# 12. WRITE ALL TABLES
# ─────────────────────────────────────────────────────────────
tables = {
    "dim_countries":         df_countries,
    "dim_product_categories":df_categories,
    "dim_products":          df_products,
    "dim_transport_modes":   df_transport_modes,
    "dim_ports":             df_ports,
    "dim_suppliers":         df_suppliers,
    "dim_supplier_products": df_supplier_products,
    "dim_customers":         df_customers,
    "dim_fx_rates":          df_fx,
    "dim_country_risk":      df_country_risk,
    "fact_purchase_orders":  df_po,
    "fact_po_state_log":     df_po_states,
    "fact_sales_orders":     df_so,
    "fact_supplier_audits":  df_audits,
    "fact_port_congestion":  df_port_congestion,
}

for name, df in tables.items():
    path = f"{OUTPUT_DIR}/{name}.csv"
    df.to_csv(path, index=False)
    print(f"  ✓ {name:35s} {len(df):>7,} rows  →  {path}")

  ✓ dim_countries                            15 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_countries.csv
  ✓ dim_product_categories                    8 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_product_categories.csv
  ✓ dim_products                             32 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_products.csv
  ✓ dim_transport_modes                       6 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_transport_modes.csv
  ✓ dim_ports                                13 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_ports.csv
  ✓ dim_suppliers                            27 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_suppliers.csv
  ✓ dim_supplier_products                   112 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_supplier_products.csv
  ✓ dim_customers                            60 rows  →  C:\Courses\Innopolis\2026\ВЭД\Данные2/dim_customers.csv
  ✓ dim_fx_rates                          2,557 rows  →  C:\Courses\Innopolis\

In [35]:
tables["fact_port_congestion"]

,pc_id,date,port_id,congestion_index,vessels_waiting
0,1,2019-01-01,CNSHA,0.1366,2
1,2,2019-01-02,CNSHA,0.1560,4
2,3,2019-01-03,CNSHA,0.1429,10
3,4,2019-01-04,CNSHA,0.0987,11
4,5,2019-01-05,CNSHA,0.1351,12
...,...,...,...,...,...
33236,33237,2025-12-27,RUNVS,0.4141,28
33237,33238,2025-12-28,RUNVS,0.4819,43
33238,33239,2025-12-29,RUNVS,0.4429,31
33239,33240,2025-12-30,RUNVS,0.5584,47


In [ ]:
# ─────────────────────────────────────────────────────────────
# 13. DATA DICTIONARY
# ─────────────────────────────────────────────────────────────
DICTIONARY = """
# International Trade Dataset — Data Dictionary
## Company: GlobalTrade LLC, Moscow, Russia | Period: 2019-01-01 to 2024-12-31

### DIMENSION TABLES

#### dim_countries
| Column | Type | Description |
|--------|------|-------------|
| country_id | CHAR(2) | ISO alpha-2 country code (PK) |
| country_name | VARCHAR | Full country name |
| region | VARCHAR | Geographic region |
| country_risk_base | FLOAT [0-1] | Base political/credit risk score |
| sanctions_risk | FLOAT [0-1] | International sanctions exposure |
| port_efficiency_base | FLOAT [0-1] | Port operational efficiency baseline |
| avg_customs_days | INT | Average customs clearance days |
| currency_code | CHAR(3) | ISO 4217 currency code |
| language | VARCHAR | Primary business language |
| wto_member | BOOL | WTO membership flag |

#### dim_product_categories
| Column | Type | Description |
|--------|------|-------------|
| category_id | INT | Category PK |
| category_name | VARCHAR | Category name |
| category_description | TEXT | Description |

#### dim_products
| Column | Type | Description |
|--------|------|-------------|
| product_id | INT | Product PK |
| product_name | VARCHAR | Product name |
| category_id | INT | FK → dim_product_categories |
| unit | VARCHAR | Unit of measure |
| base_price_usd | FLOAT | Reference price in USD |
| weight_kg_per_unit | FLOAT | Weight per unit in kg |
| hs_code | CHAR(6) | HS tariff code |
| perishable | BOOL | Whether product is perishable |

#### dim_transport_modes
| Column | Type | Description |
|--------|------|-------------|
| transport_mode_id | INT | Mode PK |
| mode_name | VARCHAR | Transport mode name |
| avg_transit_days | INT | Average transit time in days |
| cost_multiplier | FLOAT | Relative cost vs baseline |
| delay_probability | FLOAT [0-1] | Probability of delay |
| uses_port | BOOL | Whether shipment passes through a seaport |

#### dim_ports
| Column | Type | Description |
|--------|------|-------------|
| port_id | CHAR(5) | UN/LOCODE port code (PK) |
| port_name | VARCHAR | Port name |
| country_id | CHAR(2) | FK → dim_countries |
| efficiency_base | FLOAT [0-1] | Baseline port efficiency |
| annual_teu_capacity | INT | Annual container throughput capacity (TEU) |

#### dim_suppliers
| Column | Type | Description |
|--------|------|-------------|
| supplier_id | INT | Supplier PK |
| supplier_name | VARCHAR | Company name |
| country_id | CHAR(2) | FK → dim_countries |
| city | VARCHAR | City |
| primary_category_id | INT | FK → dim_product_categories |
| registration_date | DATE | Date supplier was registered in system |
| years_of_relationship | FLOAT | Years of business relationship |
| reliability_score | FLOAT [0-1] | Historical reliability score |
| maturity_score | INT | Supplier maturity level (1-20) |
| payment_terms_days | INT | Standard payment terms in days |
| min_order_usd | INT | Minimum order value in USD |
| certifications | JSON | List of certifications held |
| is_fraudster | BOOL | **LABEL** — TRUE if supplier is fraudulent |
| active | BOOL | Whether supplier is currently active |

#### dim_supplier_products
| Column | Type | Description |
|--------|------|-------------|
| sp_id | INT | PK |
| supplier_id | INT | FK → dim_suppliers |
| product_id | INT | FK → dim_products |
| supplier_price_usd | FLOAT | Supplier's quoted price in USD |
| lead_time_days | INT | Production/sourcing lead time |
| moq | INT | Minimum order quantity |

#### dim_customers
| Column | Type | Description |
|--------|------|-------------|
| customer_id | INT | Customer PK |
| customer_name | VARCHAR | Russian company name |
| city | VARCHAR | Russian city |
| latitude / longitude | FLOAT | Geographic coordinates |
| industry | VARCHAR | Customer industry |
| segment | VARCHAR | Enterprise / SME / Micro |
| annual_revenue_mln_rub | INT | Annual revenue in million RUB |
| credit_limit_usd | INT | Credit limit in USD |
| inn | CHAR(10) | Russian tax identification number |

#### dim_fx_rates
| Column | Type | Description |
|--------|------|-------------|
| fx_id | INT | PK |
| date | DATE | Trading date |
| USDRUB | FLOAT | USD/RUB rate |
| USDCNY | FLOAT | USD/CNY rate |
| USDEUR | FLOAT | USD/EUR rate |
| ... | | Other currency pairs |
*Note: Rates reflect realistic trends including 2022 sanctions shock on RUB*

#### dim_country_risk
| Column | Type | Description |
|--------|------|-------------|
| cr_id | INT | PK |
| country_id | CHAR(2) | FK → dim_countries |
| quarter_start | DATE | First day of quarter |
| country_risk_score | FLOAT [0-1] | Dynamic risk score for that quarter |
| sanctions_risk_score | FLOAT [0-1] | Dynamic sanctions risk |
| political_stability | FLOAT [0-1] | Political stability index |

---

### FACT TABLES

#### fact_purchase_orders
Main transaction table. Contains intentional data quality issues for cleaning tasks.
**Regression target**: actual_delivery_days
**Classification target**: is_fraud

| Column | Type | Description |
|--------|------|-------------|
| po_id | INT | PK |
| order_date | DATE | Date order was placed |
| supplier_id | INT | FK → dim_suppliers |
| product_id | INT | FK → dim_products |
| transport_mode_id | INT | FK → dim_transport_modes |
| origin_port_id | CHAR(5) | FK → dim_ports (nullable) |
| destination_port_id | CHAR(5) | FK → dim_ports (nullable) |
| quantity | INT | Units ordered |
| unit_price_usd | FLOAT | Price per unit in USD |
| total_value_usd | FLOAT | Total order value in USD |
| currency_code | CHAR(3) | Supplier's invoicing currency |
| fx_rate_to_usd | FLOAT | Exchange rate on order date |
| total_value_local | FLOAT | Total in supplier's currency |
| freight_cost_usd | FLOAT | Freight cost |
| insurance_cost_usd | FLOAT | Insurance cost |
| customs_duty_usd | FLOAT | Estimated customs duty |
| incoterm | VARCHAR | Trade term (EXW/FOB/CIF/etc.) |
| payment_method | VARCHAR | Payment instrument |
| expected_delivery_days | INT | Planned delivery days |
| actual_delivery_days | INT | **Regression target** (nullable if cancelled) |
| port_congestion_at_order | FLOAT [0-1] | Port congestion when order was placed |
| shock_index_at_order | FLOAT [0-1] | Macro shock severity at order date |
| supplier_reliability | FLOAT [0-1] | Reliability score at time of order |
| final_state | VARCHAR | Final order state |
| is_fraud | BOOL | **Classification target** |
| fraud_type | VARCHAR | overinvoice / underinvoice / phantom (nullable) |
*Data quality issues: missing actual_delivery_days, outliers in value, duplicates, negative quantities, date format inconsistencies in sales*

#### fact_po_state_log
State machine log for each purchase order. Suitable for LSTM sequence modelling.

| Column | Type | Description |
|--------|------|-------------|
| state_event_id | INT | PK |
| po_id | INT | FK → fact_purchase_orders |
| state | VARCHAR | Order state name |
| state_date | DATE | Date state was entered |
| duration_days | INT | Days spent in this state |
*States: Draft → Confirmed → Supplier_Processing → Shipped → In_Transit → Customs_Clearance → Delivered/Cancelled*

#### fact_sales_orders
Sales from company to Russian customers.

| Column | Type | Description |
|--------|------|-------------|
| so_id | INT | PK |
| sale_date | DATE/VARCHAR | Sale date (intentional format inconsistency injected) |
| po_id | INT | FK → fact_purchase_orders (source stock) |
| customer_id | INT | FK → dim_customers |
| product_id | INT | FK → dim_products |
| quantity | INT | Units sold |
| unit_price_usd | FLOAT | Selling price per unit |
| discount_pct | FLOAT | Discount applied |
| total_value_usd | FLOAT | Net revenue in USD |
| total_value_rub | FLOAT | Net revenue in RUB |
| fx_rate_usd_rub | FLOAT | USD/RUB on sale date |
| payment_terms_days | INT | Days until payment due |
| delivery_city | VARCHAR | Russian delivery city |

#### fact_supplier_audits
Periodic supplier audit records.

| Column | Type | Description |
|--------|------|-------------|
| audit_id | INT | PK |
| supplier_id | INT | FK → dim_suppliers |
| audit_date | DATE | Audit date |
| audit_type | VARCHAR | Financial/Quality/Compliance/On-site |
| auditor | VARCHAR | Auditor name |
| findings_count | INT | Number of findings |
| audit_score | FLOAT [0-1] | Overall audit score |
| passed | BOOL | Whether supplier passed |
| notes | TEXT | Audit notes |

#### fact_port_congestion
Daily port congestion time series. Suitable for time series / forecasting tasks.

| Column | Type | Description |
|--------|------|-------------|
| pc_id | INT | PK |
| date | DATE | Date |
| port_id | CHAR(5) | FK → dim_ports |
| congestion_index | FLOAT [0-1] | Congestion level (0=clear, 1=severe) |
| vessels_waiting | INT | Number of vessels waiting |
*Seasonal patterns: Q4 peak (Oct-Dec), Chinese New Year (Jan-Feb) relief then surge. 2021 Suez spike. 2022 sanctions shock.*

---

### REALISTIC PATTERNS & STRUCTURES

**Fraud (Classification)**
- ~15-20% of suppliers are fraudsters
- Fraud types: overinvoice (price 30-150% above market), underinvoice (40-80% below), phantom (cancelled, no delivery)
- Correlated features: low audit_score, high findings_count, price far from market, frequent cancellations

**Delivery Time (Regression)**
- Key drivers: transport_mode, supplier_country, port_congestion_at_order, shock_index_at_order, supplier_reliability, customs days
- Non-linear: interactions between congestion and transit mode

**Port Congestion (Time Series)**
- Seasonality: sinusoidal Q4 peak, Feb trough
- Trend: increasing 2019-2022, stabilizes 2023+
- Shocks: COVID 2020, Suez 2021, Sanctions 2022

**FX Rates (Time Series)**
- USDRUB: strong depreciation 2022+, high volatility
- USDTRY: continuous depreciation (~25%/year)
- USDAED: pegged, almost flat
- All currencies: shock sensitivity proportional to geopolitical exposure

**LSTM (Order State Sequences)**
- 7-state machine, duration in each state varies with supplier reliability and congestion
- Fraudulent orders tend to cancel early (at Confirmed or Shipped state)

**Data Quality Issues Injected**
- Missing values: ~3-4% in key columns
- Outliers: ~1% extreme values in price/value columns
- Duplicates: ~0.5-0.8% duplicate rows
- Negative quantities: 5 rows with sign error
- Date format inconsistencies: ~1% of sale_dates in DD/MM/YYYY format
- Case inconsistencies: ~1% lowercase incoterms
"""

with open(f"{OUTPUT_DIR}/DATA_DICTIONARY.md", "w") as f:
    f.write(DICTIONARY)

print(f"\n✓ Data dictionary saved.")
print(f"\n{'='*60}")
print("DATASET SUMMARY")
print('='*60)
for name, df in tables.items():
    print(f"  {name:35s} {len(df):>8,} rows  x  {len(df.columns):>3} cols")
print(f"\n  Output: {OUTPUT_DIR}/")
print('='*60)